In [1]:
import dhg
import torch
hg = dhg.Hypergraph(5, [(0, 1, 2), (1, 3, 2), (1, 2), (0, 3, 4)]) # type: ignore

/home/zhoutong/projects/dhg/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X = torch.tensor(([[0.6460, 0.0247],
                       [0.9853, 0.2172],
                       [0.7791, 0.4780],
                       [0.0092, 0.4685],
                       [0.9049, 0.6371]]))

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dhg
from dhg.data import Cora

# 1. 准备数据
data = Cora()
X, y = data['features'], data['labels']
train_mask, test_mask = data['train_mask'], data['test_mask']

# 构建初始超图 (1-hop)
base_g = dhg.Graph(data['num_vertices'], data['edge_list'])
hg = dhg.Hypergraph.from_graph_kHop(base_g, k=1)

# 2. 定义动态 HyperGCN 模型
class DynamicHyperGCN(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch):
        super().__init__()
        # 预转换层：将稀疏词袋映射到稠密特征空间
        self.embedding = nn.Linear(in_ch, hid_ch)
        # 两层图卷积
        self.conv1 = dhg.nn.GCNConv(hid_ch, hid_ch)
        self.conv2 = dhg.nn.GCNConv(hid_ch, out_ch)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, hg):
        # A. 提取深层特征用于“造图”
        x_emb = F.relu(self.embedding(x))
        
        # B. 动态演化：基于当前学到的特征，重新构建 HyperGCN 图结构
        # 这里 with_mediator=True 能保证连通性
        with torch.no_grad():
            g_dynamic = dhg.Graph.from_hypergraph_hypergcn(hg, x_emb, with_mediator=True)
            g_dynamic = g_dynamic.to(x.device)

        # C. 在动态生成的图上进行特征聚合
        x1 = F.relu(self.conv1(x_emb, g_dynamic))
        x1 = self.dropout(x1)
        x2 = self.conv2(x1, g_dynamic)
        
        return x2

# 3. 训练配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DynamicHyperGCN(X.shape[1], 128, data['num_classes']).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

X, y = X.to(device), y.to(device)
hg = hg.to(device)

# 4. 训练循环
print(f"{'Epoch':<8} | {'Loss':<10} | {'Test Acc':<10}")
print("-" * 35)

for epoch in range(101):
    model.train()
    optimizer.zero_grad()
    out = model(X, hg)
    loss = F.cross_entropy(out[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            pred = out.argmax(dim=1)
            acc = (pred[test_mask] == y[test_mask]).float().mean()
            print(f"{epoch:<8} | {loss.item():<10.4f} | {acc:<10.2%}")

Epoch    | Loss       | Test Acc  
-----------------------------------
0        | 1.9480     | 17.00%    
10       | 1.9283     | 10.50%    
20       | 1.8686     | 14.40%    
30       | 1.7188     | 18.60%    
40       | 1.6525     | 18.70%    
50       | 1.6203     | 22.70%    
60       | 1.7182     | 21.80%    
70       | 1.6379     | 22.10%    
80       | 1.6540     | 21.80%    
90       | 1.6237     | 21.50%    
100      | 1.6117     | 21.40%    


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dhg
from dhg.data import Cooking200
from dhg.nn import HGNNConv

# 1. 加载数据
print("正在加载 Cooking200 数据集...")
data = Cooking200()
num_v = data['num_vertices']
num_classes = data['num_classes']
y = data['labels']
train_mask, test_mask = data['train_mask'], data['test_mask']

# 2. 构建超图并【手动添加自环】
# 因为 dhg 的 Hypergraph 对象没有 add_self_loops 属性，我们直接在初始化时拼接
# data['edge_list'] 是原始菜谱，[(i,) for i in range(num_v)] 是每个食材的自环
all_edges = data['edge_list'] + [(i,) for i in range(num_v)]
hg = dhg.Hypergraph(num_v, all_edges)

# 3. 定义带 Embedding 层的 HGNN 模型
class CookingHGNN(nn.Module):
    def __init__(self, num_v, emb_dim, hid_dim, out_dim):
        super().__init__()
        # 核心：每个食材节点从随机向量开始，通过训练学习其“身份”
        self.v_embedding = nn.Embedding(num_v, emb_dim)
        
        # 两层超图卷积
        self.conv1 = HGNNConv(emb_dim, hid_dim)
        self.conv2 = HGNNConv(hid_dim, out_dim)
        self.dropout = nn.Dropout(0.5)

    def forward(self, v_ids, hg):
        # 1. 获取可学习的食材特征
        x = self.v_embedding(v_ids)
        
        # 2. 第一层卷积 + 激活 + 随机失活
        x = F.relu(self.conv1(x, hg))
        x = self.dropout(x)
        
        # 3. 第二层卷积得到分类输出
        x = self.conv2(x, hg)
        return x

# 4. 训练配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# emb_dim=128：给每个食材 128 维的特征空间来表达自己
model = CookingHGNN(num_v, 128, 256, num_classes).to(device)

# 优化器建议：训练 Embedding 层时，初始学习率不宜过高，0.005 比较稳妥
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

# 准备训练数据
v_ids = torch.arange(num_v).to(device)
y, hg = y.to(device), hg.to(device)

# 5. 训练循环
print(f"{'Epoch':<8} | {'Loss':<10} | {'Test Acc':<10}")
print("-" * 35)

best_acc = 0
for epoch in range(201):
    model.train()
    optimizer.zero_grad()
    
    out = model(v_ids, hg)
    loss = F.cross_entropy(out[train_mask], y[train_mask])
    
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            pred = out.argmax(dim=1)
            acc = (pred[test_mask] == y[test_mask]).float().mean()
            print(f"{epoch:<8} | {loss.item():<10.4f} | {acc:<10.2%}")
            if acc > best_acc:
                best_acc = acc

print("-" * 35)
print(f"🎉 实验完成！最高测试准确率: {best_acc:.2%}")

# 6. (可选) 可视化 Embedding 的聚类趋势
# 如果有兴趣，可以观察 Test Acc 是否明显超过了之前的 22.88%

正在加载 Cooking200 数据集...
Epoch    | Loss       | Test Acc  
-----------------------------------
0        | 2.9969     | 5.03%     
20       | 2.8988     | 4.37%     
40       | 2.6997     | 7.07%     
60       | 2.3751     | 7.97%     
80       | 2.2196     | 9.37%     
100      | 2.2524     | 10.82%    
120      | 2.1237     | 12.49%    
140      | 2.2110     | 14.05%    
160      | 2.2425     | 12.85%    
180      | 2.3285     | 13.57%    
200      | 2.4304     | 13.32%    
-----------------------------------
🎉 实验完成！最高测试准确率: 14.05%
